# PyPSA-Eur Tutorial

This notebook walks through the full PyPSA-Eur workflow: setup, running the model, and analysing results from a solved sector-coupled network.

## Part 1 — Setup & running the workflow

### 1.1 Install Pixi (package manager)

```bash
curl -fsSL https://pixi.sh/install.sh | bash
source ~/.bashrc  # or restart terminal
```

### 1.2 Clone PyPSA-Eur

```bash
git clone https://github.com/PyPSA/pypsa-eur.git
cd pypsa-eur
```

### 1.3 Install the environment

```bash
pixi install
```

Reads `pixi.toml` and installs all dependencies into `.pixi/envs/default/`. Takes a few minutes on first run.

### 1.4 Configure

```bash
cp config/config.default.yaml config/config.yaml
```

Edit `config/config.yaml` to set countries, number of clusters, time horizon, etc.

### 1.5 Run the workflow

```bash
pixi run snakemake -c all all -j 4
```

Resume after a crash:

```bash
pixi run snakemake -c all all -j 4 --rerun-incomplete
```

---

### Key fixes needed for short simulations (< 3 days)

Add these to `config/config.yaml`:

```yaml
sector:
  regional_co2_sequestration_potential:
    enable: false
  district_heating:
    supply_temperature_approximation:
      rolling_window_ambient_temperature: 48   # must be <= n_hours

plotting:
  costs_threshold: 0.001
  energy_threshold: 0.1
```

## Part 2 — Loading and exploring the network

Results are stored as NetCDF files under `pypsa-eur/results/networks/`.
The filename encodes the configuration: `base_s_{clusters}_{opts}_{planning_horizon}.nc`

In [1]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

n = pypsa.Network("pypsa-eur/results/networks/base_s_1___2050.nc")
print(n)

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, storage_units, stores, sub_networks


PyPSA Network 'Unnamed Network'


### Components present in this network

In [22]:
for component in n.components:
    df = getattr(n, component.list_name)
    if len(df) > 0:
        print(f"{component.name:20s}: {len(df):4d} rows")

Bus                 :   33 rows
Carrier             :  115 rows
Generator           :   20 rows
Load                :   20 rows
Link                :   65 rows
Store               :   16 rows
StorageUnit         :    2 rows
LineType            :   59 rows
TransformerType     :   14 rows
GlobalConstraint    :    3 rows
SubNetwork          :   33 rows


In [ ]:
# Load shedding generators                                                       
load_shed = n.generators[n.generators.carrier == "load"]  
                                                                                        
# Hours where load shedding occurred                                                     
p_shed = n.generators_t.p[load_shed.index]
lol_hours = (p_shed > 0).any(axis=1)                                                     
                                                        
print(f"LOLE: {lol_hours.sum()} hours/year")                                             
print(f"EENS: {p_shed.sum().sum():.1f} MWh/year")                                                    
                                                        
# Critical hours (low renewables + high demand)                                  
renewable_gen = n.generators_t.p[                         
    n.generators[n.generators.carrier.isin(["solar","onwind","offwind-ac"])].index       
].sum(axis=1)                                                                            
demand = n.loads_t.p_set.sum(axis=1)
residual_load = demand - renewable_gen                                                   
critical = residual_load.nlargest(10)                     
print("Top 10 critical hours:")                                                          
print(critical)  

LOLE: 0 hours/year
EENS: 0.0 MWh/year
Top 10 critical hours:
snapshot
2013-01-02 08:00:00    110318.953684
2013-01-02 07:00:00    109227.032612
2013-01-02 17:00:00    108808.460119
2013-01-02 16:00:00    107401.771164
2013-01-02 09:00:00    105115.513214
2013-01-02 10:00:00    103451.645691
2013-01-02 18:00:00    103094.487484
2013-01-02 06:00:00    101954.810350
2013-01-02 11:00:00    101843.767039
2013-01-02 15:00:00    101161.686102
dtype: float64
